# AI-Based Rubik's Cube Solver - Thesis Training
**Edward Ogbei | MSc Artificial Intelligence | Politechnika Czestochowa | 2026**

---

This notebook trains the neural network models for my MSc thesis on a GPU, which is
much faster than running it on my laptop. Everything runs from top to bottom without
any manual steps in between.

**Before you start:** go to `Runtime > Change runtime type`, pick **T4 GPU**, and save.
Then just click `Runtime > Run all` and wait.

Here is what each stage does:

| Stage | Description | Time |
|-------|-------------|------|
| 1 | Check GPU, download the project code, install packages | ~2 min |
| 2 | Train small, medium, and large models to see where each one hits its limit | ~20-25 min |
| 3 | Train the final production model with the best settings | ~12-15 min |
| 4 | Run the benchmark comparing AI against Kociemba at depths 1 to 10 | ~5-8 min |
| 5 | Download 3 files to send back | instant |

Total time is roughly **40 to 50 minutes** on a free T4 GPU.

---
## Stage 1 - Setup

First check that a GPU is actually available, then pull the project code from GitHub
and install everything it needs.

In [ ]:
# Make sure we have a GPU. If this fails, go to Runtime > Change runtime type > T4 GPU
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch:', torch.__version__)

In [ ]:
import os

repo_name = 'AI-Based_Optimal_Solver_for_the_3x3_Rubiks_Cube'
repo_url  = 'https://github.com/ogbeiedward/AI-Based_Optimal_Solver_for_the_3x3_Rubiks_Cube.git'

# Only clone if we haven't already
if not os.path.exists(repo_name):
    !git clone {repo_url}

%cd {repo_name}
!git pull  # pick up any recent changes

!pip install -r requirements.txt -q
!pip install reportlab -q

print('All done.')

---
## Stage 2 - Ablation Study (Small, Medium, Large Models)

This is the core experiment for the thesis. We train three different-sized models
using curriculum learning and watch where each one gives up.

The rule is simple: the model starts at depth 1 (one scramble move) and only moves
to depth 2 once it can solve at least 80% of test scrambles at depth 1. Same for
every level after that. If it can't hit 80%, training stops at that depth - that is
the honest capacity ceiling for that model size.

What we're training:
- **Small model** - two hidden layers of 128 neurons, ~42k parameters, 15k samples per depth, 50 epochs
- **Medium model** - 256 then 128 neurons, ~100k parameters, 30k samples per depth, 75 epochs
- **Large model** - 256, 256 then 128 neurons, ~184k parameters, 50k samples per depth, 100 epochs

Max depth we push to: **7**. Going beyond that needs far more data and time than
is realistic for this architecture.

In [ ]:
import sys, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, '.')
from core.cube import CubieCube
from core.state_encoder import encode_state, ENCODING_DIM, NUM_MOVES, MOVE_TO_INDEX
from solvers.kociemba_solver import solve_with_kociemba
from utils.scramble import generate_scramble_at_depth

device = torch.device('cuda')
print('Training device:', device)

In [ ]:
# The neural network. Hidden layer sizes are passed in as a list so we
# can swap between small/medium/large without duplicating code.
class FlexMLP(nn.Module):
    def __init__(self, hidden_sizes):
        super().__init__()
        layers, in_dim = [], ENCODING_DIM
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        layers.append(nn.Linear(in_dim, NUM_MOVES))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def predict_move(self, x):
        self.eval()
        with torch.no_grad():
            if x.dim() == 1:
                x = x.unsqueeze(0)
            probs = torch.softmax(self.net(x), dim=1)
            idx   = torch.argmax(probs, dim=1).item()
            return idx, probs[0, idx].item()

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Build training samples by scrambling cubes and recording what Kociemba
# says the best next move is at each step along the solution path.
def generate_data(n_samples, max_depth, seed=None):
    rng    = random.Random(seed)
    states = []
    labels = []

    for _ in range(n_samples):
        depth = rng.randint(1, max_depth)
        scr   = generate_scramble_at_depth(depth, seed=rng.randint(0, 2**31 - 1))
        cube  = CubieCube()
        cube.apply_sequence(scr)

        if cube.is_solved():
            continue

        res = solve_with_kociemba(cube)
        if res['error'] or not res['validated']:
            continue

        current = cube.copy()
        for move in res['solution'].split():
            idx = MOVE_TO_INDEX.get(move)
            if idx is None:
                continue
            states.append(encode_state(current))
            labels.append(idx)
            current.apply_move(move)

    return (np.array(states, dtype='float32'),
            np.array(labels, dtype='int64'))

In [ ]:
# Test the model on fresh scrambles it has never seen.
# Returns the fraction it solves correctly.
def eval_solve_rate(model, depth, n=50, seed=0):
    from solvers.ai_solver import solve_with_ai
    rng    = random.Random(seed)
    solved = 0
    model.eval()
    for _ in range(n):
        scr  = generate_scramble_at_depth(depth, seed=rng.randint(0, 2**31 - 1))
        cube = CubieCube()
        cube.apply_sequence(scr)
        result = solve_with_ai(cube, model, device=device)
        if result['solved']:
            solved += 1
    return solved / n

In [ ]:
# The main curriculum training loop.
# Starts at depth 1 and only advances when the model hits the solve-rate threshold.
# If it can't hit the threshold, it stops - that tells us the model's hard limit.
def train_curriculum(model, samples_per_depth, epochs_per_depth,
                     max_depth=7, threshold=0.80, batch_size=512, lr=1e-3, seed=42):

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    rng       = random.Random(seed)

    loss_history  = []
    depth_results = []

    for depth in range(1, max_depth + 1):
        print(f'  Depth {depth} of {max_depth} - generating {samples_per_depth:,} samples...')
        states, labels = generate_data(samples_per_depth, depth,
                                       seed=rng.randint(0, 2**31 - 1))

        dataset = TensorDataset(torch.tensor(states), torch.tensor(labels))
        loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        print(f'  Got {len(dataset):,} samples. Training for {epochs_per_depth} epochs...')

        model.train()
        for epoch in range(epochs_per_depth):
            total_loss = 0.0
            num_batches = 0
            for xs, ys in loader:
                xs, ys = xs.to(device), ys.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xs), ys)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                num_batches += 1
            avg_loss = total_loss / max(num_batches, 1)
            loss_history.append(avg_loss)
            if (epoch + 1) % 25 == 0:
                print(f'    Epoch {epoch + 1}/{epochs_per_depth}  loss = {avg_loss:.4f}')

        # Check if the model has actually learned this depth
        solve_rate = eval_solve_rate(model, depth, n=50,
                                     seed=rng.randint(0, 2**31 - 1))

        if solve_rate >= threshold:
            print(f'  Solve rate at depth {depth}: {solve_rate:.1%} - passed, moving to next depth')
        else:
            print(f'  Solve rate at depth {depth}: {solve_rate:.1%} - did not reach {threshold:.0%}, stopping here')

        depth_results.append({
            'depth': depth,
            'solve_rate': solve_rate,
            'num_training_samples': len(dataset),
        })

        if solve_rate < threshold:
            print(f'  This model has hit its ceiling at depth {depth}.')
            print(f'  To go further we would need more parameters, more data, or both.')
            break

    return loss_history, depth_results

In [ ]:
# Three model sizes to compare. Same training loop, different capacity.
EXPERIMENTS = [
    {
        'label':   'Small (~42k params)',
        'short':   'small',
        'hidden':  [128, 128],
        'samples': 15_000,
        'epochs':  50,
        'color':   '#b45309',
    },
    {
        'label':   'Medium (~100k params)',
        'short':   'medium',
        'hidden':  [256, 128],
        'samples': 30_000,
        'epochs':  75,
        'color':   '#2563eb',
    },
    {
        'label':   'Large (~184k params)',
        'short':   'large',
        'hidden':  [256, 256, 128],
        'samples': 50_000,
        'epochs':  100,
        'color':   '#166534',
    },
]

all_results = []

for cfg in EXPERIMENTS:
    print(f"\n{'='*55}")
    print(f"  {cfg['label']}")
    print(f"  {cfg['samples']:,} samples/depth, {cfg['epochs']} epochs/depth")
    print(f"{'='*55}")

    model = FlexMLP(cfg['hidden'])
    print(f'  Parameters: {model.n_params:,}')

    t0 = time.time()
    loss_hist, depth_res = train_curriculum(
        model,
        samples_per_depth=cfg['samples'],
        epochs_per_depth=cfg['epochs'],
        max_depth=7,
        threshold=0.80,
        batch_size=512,
        seed=42,
    )
    elapsed = time.time() - t0
    print(f'  Training finished in {elapsed:.0f}s')

    # After training, test it at every depth to get the final numbers for the charts
    print('  Testing final solve rates...')
    final_rates = {}
    for d in range(1, 8):
        rate = eval_solve_rate(model, d, n=50, seed=42 + d)
        final_rates[d] = rate
        print(f'    Depth {d}: {rate:.1%}')

    all_results.append({
        'label':              cfg['label'],
        'short':              cfg['short'],
        'color':              cfg['color'],
        'n_params':           model.n_params,
        'samples_per_depth':  cfg['samples'],
        'loss_history':       [float(x) for x in loss_hist],
        'depth_results':      depth_res,
        'final_solve_rates':  {str(k): float(v) for k, v in final_rates.items()},
        'train_time':         elapsed,
    })

os.makedirs('experiments/plots', exist_ok=True)
with open('experiments/plots/experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('\nAblation results saved.')

---
## Stage 3 - Production Model Training

This trains the actual model that goes into the web app.
It uses the same large architecture as above but we push the sample count and
epoch count to their maximum to get the best possible result.

It will advance through depths 1 to 7 as long as it keeps clearing 80%.
Wherever it stops is where it stops - no faking it.

In [ ]:
print('Starting production model training...')
print('Architecture: [256, 256, 128]  |  50k samples/depth  |  100 epochs/depth  |  max depth 7')
print()

prod_model = FlexMLP([256, 256, 128])
print(f'Parameters: {prod_model.n_params:,}')

t0 = time.time()

prod_loss_hist, prod_depth_results = train_curriculum(
    prod_model,
    samples_per_depth=50_000,
    epochs_per_depth=100,
    max_depth=7,
    threshold=0.80,
    batch_size=512,
    lr=1e-3,
    seed=99,
)

print(f'\nFinished in {time.time() - t0:.0f}s')

# Save model weights
os.makedirs('data/models', exist_ok=True)
torch.save(prod_model.state_dict(), 'data/models/ai_solver.pt')
print('Saved to data/models/ai_solver.pt')

# Quick summary
print()
print('Summary:')
for row in prod_depth_results:
    filled = int(row['solve_rate'] * 20)
    bar    = '#' * filled + '.' * (20 - filled)
    print(f"  Depth {row['depth']}: {row['solve_rate']:.1%}  [{bar}]")

---
## Stage 4 - Benchmark (AI vs Kociemba, Depths 1 to 10)

Now we run 50 test scrambles at each depth from 1 to 10 and record:
- How often each solver succeeds
- How long it takes in milliseconds
- How many moves the solution uses

We test four solvers: Kociemba, AI Greedy, AI Beam Search with width 3, and width 5.
This gives us the numbers for the comparison tables and timing charts in the thesis.

In [ ]:
from solvers.ai_solver import solve_with_ai, solve_with_beam_search

TRIALS = 50
rng    = random.Random(7)

koc_rows = []
g_rows   = []
b3_rows  = []
b5_rows  = []

prod_model.eval()

print('Running benchmark...')
print(f'{"Depth":>6}  {"Kociemba":>10}  {"Greedy":>8}  {"Beam w3":>8}  {"Beam w5":>8}')
print('-' * 50)

for depth in range(1, 11):
    koc_s = koc_t = koc_m = 0
    g_s   = g_t   = g_m   = 0
    b3_s  = b3_t  = b3_m  = 0
    b5_s  = b5_t  = b5_m  = 0

    for _ in range(TRIALS):
        scr  = generate_scramble_at_depth(depth, seed=rng.randint(0, 2**31 - 1))
        cube = CubieCube()
        cube.apply_sequence(scr)

        r = solve_with_kociemba(cube)
        if r['error'] is None:
            koc_s += 1; koc_t += r['solve_time'] * 1000; koc_m += r['num_moves']

        r = solve_with_ai(cube, prod_model, device=device)
        if r['solved']:
            g_s += 1; g_t += r['solve_time'] * 1000; g_m += r['num_moves']

        r = solve_with_beam_search(cube, prod_model, beam_width=3, device=device)
        if r['solved']:
            b3_s += 1; b3_t += r['solve_time'] * 1000; b3_m += r['num_moves']

        r = solve_with_beam_search(cube, prod_model, beam_width=5, device=device)
        if r['solved']:
            b5_s += 1; b5_t += r['solve_time'] * 1000; b5_m += r['num_moves']

    def make_row(s, t, m):
        return {
            'depth': depth,
            'rate':  round(s / TRIALS, 3),
            'moves': round(m / s, 2) if s else 0,
            'ms':    round(t / s, 2) if s else 0,
        }

    koc_rows.append(make_row(koc_s, koc_t, koc_m))
    g_rows.append(make_row(g_s,   g_t,   g_m))
    b3_rows.append(make_row(b3_s,  b3_t,  b3_m))
    b5_rows.append(make_row(b5_s,  b5_t,  b5_m))

    print(f'  {depth:>2}      {koc_s:>4}/{TRIALS}    {g_s:>4}/{TRIALS}   {b3_s:>4}/{TRIALS}   {b5_s:>4}/{TRIALS}')

os.makedirs('docs', exist_ok=True)
with open('docs/eval_results.json', 'w') as f:
    json.dump({
        'kociemba': koc_rows,
        'greedy':   g_rows,
        'beam_w3':  b3_rows,
        'beam_w5':  b5_rows,
    }, f, indent=2)

print('\nBenchmark results saved to docs/eval_results.json')

---
## Stage 5 - Download the Results

Run these three cells one by one. Each one will prompt a file download in your browser.

These are the three files to send back:

| File | What is inside |
|------|----------------|
| `experiment_results.json` | Loss curves and solve rates for all three model sizes, depth by depth |
| `eval_results.json` | Benchmark numbers comparing Kociemba vs the AI solvers at depths 1 to 10 |
| `ai_solver.pt` | The trained model weights for the web app |

In [ ]:
from google.colab import files

# Download 1 of 3 - training results
files.download('experiments/plots/experiment_results.json')

In [ ]:
# Download 2 of 3 - benchmark results
files.download('docs/eval_results.json')

In [ ]:
# Download 3 of 3 - the trained model
files.download('data/models/ai_solver.pt')

---
## That is it

Once you have all three files, send them back. I will:

1. Drop the new model into `data/models/ai_solver.pt` and restart the web server
2. Generate all the thesis figures from the real training data - loss curves showing each depth stage, solve rate bars by model size, timing comparison
3. Build the full improved PDF with proper equations and professional charts
4. Update the benchmark table in the web UI

The training here used the GPU properly, so the curves will be much cleaner
and the model will solve harder scrambles than the version trained on your laptop.